# Push-up form classifier - Colab training runner

The Kaggle dataset `mohamadashrafsalama/pushup` is **raw videos** (Correct/Wrong sequence/*.mp4),
not a feature CSV. So the TRAIN cell runs **MediaPipe pose extraction per video**, segments reps,
and builds rep-level features (same definitions as the app's `:core` PushUpFeatureExtractor) before
training. Extraction of ~100 videos takes a **few minutes**.

Run top to bottom. Mount Drive with the **kimgt2828** account; complete the Kaggle login form.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/health_training'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
print(DRIVE_ROOT)

In [ ]:
# Clone the model-training branch (has the push-up video pipeline). %cd /content first = re-run safe.
%cd /content
!rm -rf /content/health_trainer
!git clone --branch model-training --single-branch https://github.com/kimgt0128/health-trainer.git /content/health_trainer
%cd /content/health_trainer
# Push-up training extracts landmarks from video -> needs mediapipe + opencv (+ kagglehub).
!pip install -q mediapipe opencv-python "kagglehub[pandas-datasets]"

In [ ]:
# Kaggle auth - enter username + API token, wait for the green confirmation, then continue.
import kagglehub
kagglehub.login()

In [ ]:
# === TRAIN === downloads the dataset + the MediaPipe .task, extracts per-video landmarks, segments
# reps, builds rep-level features, trains HGB, writes 4 artifacts. ~100 videos -> a few minutes.
%cd /content/health_trainer
import os
os.environ['PYTHONPATH'] = '/content/health_trainer/ml/src'
!python ml/src/train_pushup_form_classifier.py \
    --run-dir "$RUNS_DIR/pushup_form_classifier_v1" \
    --test-size 0.2 \
    --random-state 42

In [ ]:
# Inspect artifacts written to Drive (4 files expected).
!find "$RUNS_DIR/pushup_form_classifier_v1" -maxdepth 1 -type f -print
!cat "$RUNS_DIR/pushup_form_classifier_v1/metrics_summary.json"